<div style="background:#0A005A;border-radius:18px;padding:28px 32px;color:#fff;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif">

<div style="margin-top:22px"><span style="background:#613EFF;color:#fff;padding:5px 12px;border-radius:999px;font-size:.8em;font-weight:700;letter-spacing:.06em">GUIA · NOTEBOOK</span>
<span style="color:#C9C2FF;font-size:.85em;margin-left:8px">Engenharia de Dados · Nível 0 · Fundamentos</span></div>
<h1 style="color:#fff;margin:14px 0 8px;font-size:2.1em;line-height:1.15">Preparar o Jupyter e olhar os dados</h1>
<p style="color:#E4E0FF;margin:0 0 12px;font-size:1.05em">Confira que o ambiente funciona e conheça as estações e as viagens da Pedala Aurora.</p>
<span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">cerca de 15 minutos</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">Python 3.10+</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">roda no Colab</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">dados fictícios</span>
</div>

## O que você vai fazer

- Conferir que o Python, os pacotes e a API de mentira respondem.
- Ver as estações e uma viagem, do jeito que a fonte entrega.
- Juntar um dia inteiro de viagens e responder à primeira pergunta.

A explicação completa está na aula: [Preparar o Jupyter e olhar os dados](https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/guias/engenharia-fundamentos-jupyter.md).

## Como usar este notebook

Rode as células **em ordem**, de cima para baixo: clique na célula e aperte <kbd>Shift</kbd> + <kbd>Enter</kbd>. A primeira linha de cada célula diz o que fazer com ela:

| Começa com | O que fazer |
|---|---|
| `# Rode e observe` | Código pronto. Rode, leia a saída e compare com a explicação acima dela. |
| `# Sua vez` | Complete o código onde está `None` e rode. |
| `# Confira` | Rode depois da sua vez: ela mostra `[OK]` se está certo, `[ERRO]` com o que falta, ou `[AVISO]` se você ainda não respondeu. |

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Onde rodar</b><br>No <b>Google Colab</b> (só abrir, sem instalar nada), no <b>VS Code</b> com a extensão Jupyter ou no <b>JupyterLab</b> (<code>pip install jupyterlab</code> e depois <code>jupyter lab</code>). Os dados são fictícios e ficam no próprio notebook: nada vai para fora da sua máquina.</div>

##  Preparação

Instale os pacotes, carregue os quadros de conferência e crie o cenário da Pedala Aurora.

In [ ]:
%pip install -q duckdb httpx pandas matplotlib

In [ ]:
from IPython.display import HTML, display


def mostrar(tipo, titulo, detalhe=""):
    """Mostra um quadro colorido com o resultado de uma conferência."""
    cor, rotulo = {"ok": ("#613EFF", "[OK]"), "erro": ("#F9006F", "[ERRO]"), "info": ("#0A005A", "[AVISO]")}[tipo]
    display(HTML(
        f'<div style="border-left:5px solid {cor};background:#F2F1F6;padding:10px 14px;'
        f'border-radius:8px;margin:6px 0;color:#202020"><b style="color:{cor}">{rotulo} {titulo}</b>'
        + (f"<br>{detalhe}" if detalhe else "") + "</div>"))


def conferir(resposta, certo, mensagem_certo, mensagem_errado):
    """Compara a sua resposta com a esperada e mostra um quadro verde, rosa ou de aviso."""
    if resposta is None:
        mostrar("info", "Ainda sem resposta", "Complete a célula “Sua vez” acima, rode-a e rode esta de novo.")
    elif certo:
        mostrar("ok", mensagem_certo)
    else:
        mostrar("erro", mensagem_errado)


mostrar("ok", "Quadros de conferência prontos")

A célula abaixo gera os dados e liga uma **API de mentira** aqui mesmo, no notebook. Ela está recolhida porque você não precisa ler o código para seguir; se tiver curiosidade, abra.

In [ ]:
#@title Preparar o cenário (clique duas vezes para ver o código)
# Cria os dados da Pedala Aurora e liga, aqui mesmo no notebook, uma API de mentira que os entrega.
# Os dados são sintéticos e sempre os mesmos (semente fixa). Você não precisa ler este código para seguir.
import json
import random
import threading
from datetime import date, datetime, timedelta, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

FUSO = timezone(timedelta(hours=-3))
ESTACOES = [
    {"id_estacao": "E01", "nome": "Praça Central", "bairro": "Centro", "vagas": 20},
    {"id_estacao": "E02", "nome": "Rodoviária", "bairro": "Centro", "vagas": 15},
    {"id_estacao": "E03", "nome": "Parque das Águas", "bairro": "Jardim Norte", "vagas": 12},
    {"id_estacao": "E04", "nome": "Mercado Municipal", "bairro": "Centro", "vagas": 16},
    {"id_estacao": "E05", "nome": "Universidade", "bairro": "Vila Nova", "vagas": 18},
    {"id_estacao": "E06", "nome": "Hospital Regional", "bairro": "Vila Nova", "vagas": 10},
    {"id_estacao": "E07", "nome": "Orla do Lago", "bairro": "Lagoa", "vagas": 14},
    {"id_estacao": "E08", "nome": "Estação do Metrô", "bairro": "Jardim Sul", "vagas": 20},
]
PRIMEIRO_DIA, ULTIMO_DIA = date(2026, 6, 1), date(2026, 6, 7)
POR_PAGINA = 40


def _gerar():
    sorteio = random.Random(2026)
    pesos = [5, 3, 2, 3, 4, 1, 2, 5]
    horas = list(range(6, 23))
    peso_hora = [2, 6, 9, 5, 3, 3, 4, 3, 3, 3, 4, 7, 9, 6, 3, 2, 1]
    viagens, atrasadas, numero = {}, [], 5000
    dia = PRIMEIRO_DIA
    while dia <= ULTIMO_DIA:
        quantas = sorteio.randint(85, 115) if dia.weekday() < 5 else sorteio.randint(55, 75)
        lista = []
        for _ in range(quantas):
            numero += 1
            saida, chegada = sorteio.choices(ESTACOES, pesos, k=2)
            hora = sorteio.choices(horas, peso_hora)[0]
            inicio = datetime(dia.year, dia.month, dia.day, hora, sorteio.randint(0, 59), tzinfo=FUSO)
            fim = inicio + timedelta(minutes=sorteio.randint(3, 45))
            lista.append({"id_viagem": f"V{numero}", "id_usuario": f"U{sorteio.randint(100, 399)}",
                          "estacao_saida": saida["id_estacao"], "estacao_chegada": chegada["id_estacao"],
                          "inicio": inicio.isoformat(), "fim": fim.isoformat()})
        lista.sort(key=lambda v: v["inicio"])
        viagens[dia.isoformat()] = lista
        dia += timedelta(days=1)
    # Defeitos plantados, como os de uma fonte real:
    for v in viagens["2026-06-04"][10:12]:                 # fim antes do início
        v["inicio"], v["fim"] = v["fim"], v["inicio"]
    for dia_, i in (("2026-06-02", 30), ("2026-06-04", 50), ("2026-06-06", 20)):
        viagens[dia_][i]["estacao_saida"] = "E99"           # estação fora do cadastro
    for _ in range(4):                                      # viagens de 05/06 que só chegam depois
        numero += 1
        inicio = datetime(2026, 6, 5, 21, sorteio.randint(0, 59), tzinfo=FUSO)
        atrasadas.append({"id_viagem": f"V{numero}", "id_usuario": f"U{sorteio.randint(100, 399)}",
                          "estacao_saida": "E07", "estacao_chegada": "E01",
                          "inicio": inicio.isoformat(), "fim": (inicio + timedelta(minutes=12)).isoformat()})
    return viagens, atrasadas


VIAGENS, _ATRASADAS = _gerar()
_estado = {"atrasadas_chegaram": False, "falha_temporaria": True}


def chegam_viagens_atrasadas():
    """Simula o dia seguinte: a fonte recebe 4 viagens de 05/06 que ainda não tinham sido enviadas."""
    if not _estado["atrasadas_chegaram"]:
        VIAGENS["2026-06-05"] = sorted(VIAGENS["2026-06-05"] + _ATRASADAS, key=lambda v: v["inicio"])
        _estado["atrasadas_chegaram"] = True


class _Api(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def _responder(self, status, corpo=None):
        self.send_response(status)
        if corpo is None:
            self.end_headers()
            return
        dados = json.dumps(corpo, ensure_ascii=False).encode("utf-8")
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()
        self.wfile.write(dados)

    def do_GET(self):
        url = urlparse(self.path)
        args = {k: v[0] for k, v in parse_qs(url.query).items()}
        if url.path == "/estacoes":
            return self._responder(200, ESTACOES)
        if url.path != "/viagens":
            return self._responder(404, {"erro": "caminho inexistente"})
        try:
            dia = date.fromisoformat(args.get("data", ""))
            pagina = int(args.get("pagina", "1"))
        except ValueError:
            return self._responder(400, {"erro": "use data=AAAA-MM-DD e pagina=1, 2, ..."})
        lista = VIAGENS.get(dia.isoformat(), [])
        if not lista:
            return self._responder(204)
        if dia.isoformat() == "2026-06-02" and pagina == 2 and _estado["falha_temporaria"]:
            _estado["falha_temporaria"] = False
            return self._responder(503, {"erro": "servidor ocupado, tente de novo"})
        paginas = -(-len(lista) // POR_PAGINA)
        inicio = (pagina - 1) * POR_PAGINA
        trecho = lista[inicio:inicio + POR_PAGINA]
        if dia.isoformat() == "2026-06-03" and pagina == 2:
            trecho = [lista[inicio - 1]] + trecho               # a página 2 repete o último da página 1
        self._responder(200, {"data": dia.isoformat(), "pagina": pagina, "total_paginas": paginas,
                              "total_registros": len(lista), "viagens": trecho})


_servidor = ThreadingHTTPServer(("127.0.0.1", 0), _Api)
threading.Thread(target=_servidor.serve_forever, daemon=True).start()
API = f"http://127.0.0.1:{_servidor.server_port}"
print(f"Cenário pronto: {len(ESTACOES)} estações, {sum(len(v) for v in VIAGENS.values())} viagens "
      f"de {PRIMEIRO_DIA:%d/%m} a {ULTIMO_DIA:%d/%m}. API em {API}")

##  Passo 1 de 4 · O ambiente responde

Esta célula confere a versão do Python, os pacotes e a API. Se algo aparecer em rosa, a mensagem diz o que fazer.

In [ ]:
# Rode e observe
import sys

import duckdb
import httpx
import pandas as pd

mostrar("ok" if sys.version_info >= (3, 10) else "erro", f"Python {sys.version.split()[0]}")
mostrar("ok", f"duckdb {duckdb.__version__} · httpx {httpx.__version__} · pandas {pd.__version__}")
resposta = httpx.get(API + "/estacoes", timeout=10)
mostrar("ok" if resposta.status_code == 200 else "erro", f"API respondeu {resposta.status_code}")

##  Passo 2 de 4 · As estações

O cadastro das estações muda pouco: oito estações, cada uma com nome, bairro e número de vagas.

In [ ]:
# Rode e observe
estacoes = pd.DataFrame(resposta.json())
estacoes

##  Passo 3 de 4 · Uma viagem

Cada vez que alguém retira e devolve uma bicicleta, a fonte registra uma viagem. Peça a primeira página de 01/06 e olhe a primeira viagem.

In [ ]:
# Rode e observe
corpo = httpx.get(API + "/viagens", params={"data": "2026-06-01", "pagina": 1}, timeout=10).json()
print({chave: valor for chave, valor in corpo.items() if chave != "viagens"})
corpo["viagens"][0]

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Leia assim</b><br>A resposta tem duas partes: <code>viagens</code>, com os registros desta página, e os campos de controle, que dizem quantas páginas e registros existem no dia. Cada viagem diz quem pedalou (um código, não o nome), de onde saiu, onde chegou e quando. O <code>-03:00</code> no fim do horário é o fuso de Brasília.</div>

##  Passo 4 de 4 · Um dia inteiro

Um dia tem várias páginas. O laço abaixo pede uma por uma e junta tudo numa tabela. A aula de coleta mostra o que pode dar errado nesse laço; por enquanto, só observe o resultado.

In [ ]:
# Rode e observe
viagens, pagina = [], 1
while True:
    corpo = httpx.get(API + "/viagens", params={"data": "2026-06-01", "pagina": pagina}, timeout=10).json()
    viagens.extend(corpo["viagens"])
    if pagina == corpo["total_paginas"]:
        break
    pagina += 1

dia = pd.DataFrame(viagens)
dia["inicio"] = pd.to_datetime(dia["inicio"])
print(len(dia), "viagens em", pagina, "páginas")
grafico = dia.groupby(dia["inicio"].dt.hour).size().plot.bar(
    title="Viagens por hora em 01/06", xlabel="hora do início", ylabel="viagens", color="#613EFF")

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Leia assim</b><br>Os dois picos, de manhã e no fim da tarde, são as pessoas indo e voltando do trabalho e da escola. Um gráfico simples já mostra algo que ninguém escreveu nos dados.</div>

### Sua vez

Qual estação teve **mais saídas** em 01/06? Use a tabela `dia` e a coluna `estacao_saida`. Dica: `dia["estacao_saida"].value_counts()` conta quantas vezes cada valor aparece.

In [ ]:
# Sua vez
mais_saidas = None   # troque None pelo código da estação, por exemplo "E01"
mais_saidas

In [ ]:
# Confira
from collections import Counter

esperado = Counter(v["estacao_saida"] for v in VIAGENS["2026-06-01"]).most_common(1)[0][0]
conferir(mais_saidas, mais_saidas == esperado, f"Isso: {esperado} teve mais saídas em 01/06",
         "Ainda não: conte as saídas por estação e pegue a primeira")

<details>
<summary><b>Ver uma resposta</b> (tente antes)</summary>

```python
mais_saidas = dia["estacao_saida"].value_counts().index[0]
mais_saidas
```
</details>

## O que você aprendeu

- O notebook roda em ordem, célula por célula, e cada célula guarda o que criou para as próximas.
- A fonte entrega dois tipos de dado: um **cadastro** (estações), que muda pouco, e **eventos** (viagens), que chegam todo dia.
- Uma API costuma entregar os dados em **páginas**, com campos que dizem quantas existem.

**Próxima aula:** [Prática: coletar de uma API](https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/tutoriais/engenharia-fundamentos-coleta.md).